## setup

In [1]:
import os
import sys
sys.path.insert(0,'/root/eallmancy')
sys.path.insert(0,'/root/eallmancy/experiments')
os.chdir('/root/eallmancy')

# TODO: THIS IS ULTRA HACKY, PUT API SETTINGS IN CONTROL PANEL
with open('/root/openai_api_key') as f:
    os.environ['OPENAI_API_KEY'] = f.read().replace('\n','')
with open('/root/gemini_api_key') as f:
    os.environ['GEMINI_API_KEY'] = f.read().replace('\n','')

from eallmancy.engine.llm_solution_engine_v3 import EngineV3
from eallmancy import Artifact
from eallmancy import SinglePartSolution, ProblemSpec
from eallmancy.engine.engine_trace import EngineTrace
from collections import defaultdict
import datetime
import numpy as np
import os
import re




## benchmark analyzers (EmbeddingAnalysis)

In [2]:
from eallmancy.analysis.analyzer.utils import get_codet5p_embeddings, get_openai_embeddings, cosine_sim
from eallmancy.analysis.analyzer.analyzer import Analyzer

class PureEmbeddingAnalysis:
    def __init__(self, trace, embedding_func):
        all_solns = [a.trace_id for a in trace.list if a.is_solution]
        embs = embedding_func([CodeT5PAnalysis.get_clean_program(trace[ti]) for ti in all_solns])
        self.embs = {
            ti: emb for ti, emb in zip(all_solns, embs)
        }

    @classmethod
    def get_clean_program(cls, arte):
        return cls._clean_program(arte.program)
        
    @classmethod
    def _clean_program(cls, prog):
        # find function line
        fsig_idx = prog.find('def ')
        programbody_idx = fsig_idx + prog[fsig_idx:].find('\n')+1
        return prog[programbody_idx:]
        
    def distance(self, a, b):
        return 1 - cosine_sim(self.embs[a.trace_id], self.embs[b.trace_id])
        
class CodeT5PAnalysis(PureEmbeddingAnalysis):
    def __init__(self, trace):
        super().__init__(trace, embedding_func=get_codet5p_embeddings)
        
class OpenAIAnalysis(PureEmbeddingAnalysis):
    def __init__(self, trace):
        super().__init__(trace, embedding_func=get_openai_embeddings)

## analyzer pre-caching on disk

In [3]:
%%time

from pathlib import Path
from threading import Lock
import re
import pickle as pk

TRACEVAULT_DIR = '/root/data/trace_vault'

def find_or_default(ptrn, tstr, flags=0, default=None):
    return (re.findall(ptrn, tstr, flags=flags) or [default]).pop()

def fetch_trace(vault_name, pk_filename, max_traceid=float('inf')):
    trace_fp = Path('/root/data/trace_vault') / vault_name / pk_filename
    trace = EngineTrace.load(trace_fp.__str__())
    trace.trace_list = [a for a in trace.trace_list if a.trace_id<=max_traceid]
    # trace.trace_graph = trace.trace_graph.subgraph([a.trace_id for a in trace.trace_list])
    for arte in trace.list:  # handle fudged solutions, scraped from reevo or llm4ad
        if arte.is_solution:
            if not hasattr(arte, 'program') and hasattr(arte, 'content'):
                arte.program = find_or_default(r'```[^\n]*\n(.*?)```', arte.content , flags=re.DOTALL, default=arte.content )
    return trace

def fetch_traceanalyses(vault_name, pk_filename):
    traceanalysis_fp = Path('/root/data/cached_analyses') / vault_name / pk_filename
    with open(traceanalysis_fp.__str__(), 'rb') as f:
        all_ta = pk.load(f)
    return all_ta['elmcy'], all_ta['codet5p'], all_ta['openai']

def fetch_segmanalysis(vault_name, pk_filename):
    traceanalysis_fp = Path('/root/data/cached_segm_analyses') / vault_name / pk_filename
    with open(traceanalysis_fp.__str__(), 'rb') as f:
        segmanalysis = pk.load(f)
    segmanalysis._trace.lock = Lock()
    return segmanalysis


CPU times: user 30 μs, sys: 9 μs, total: 39 μs
Wall time: 43.9 μs


In [ ]:

import polars as pl
from itertools import chain


def pct_inherited(atmpl, trace):
    tmpl_chldrn = {c for _, c in trace.graph.subgraph(atmpl.solution_members()).edges}
    return len(tmpl_chldrn)*1. / len(atmpl.solution_members())

def segm_df(vault_name):
    
    all_dfs = []
    for fn in os.listdir(TRACEVAULT_DIR+'/'+vault_name):
        print(fn)
        segm_ta = fetch_segmanalysis(vault_name, fn)
        trace = segm_ta._trace
        df = (
            pl.DataFrame(
                [(
                    a_idx,
                    a.token_length(),
                    len(a.solution_members()),
                    pct_inherited(a, trace),
                    [trace[s].fitness for s in a.solution_members()],
                ) for a_idx, a in enumerate(segm_ta.templates)]
            , schema=['idx', 'num_tokens', 'num_members', 'pct_inherited', 'fitnesses']
            , orient='row'
            )
            # .filter(pl.col.num_tokens > 10)
            # .filter(pl.col.num_members >= 4)
        
            .with_columns(filename=pl.lit(fn))
            .with_columns(
                pl.col.filename.str.extract(r"[0-9]+-[0-9]+_[^_]+_[^_]+_([^\.,^\-]+).*\.pk", 1).alias('problem'),
                pl.col.filename.str.extract(r"[0-9]+-[0-9]+_[^_]+_([^_]+)_[^\.,^\-]+.*\.pk", 1).alias('algo')
            )
        )
        all_dfs.append(df)
    return pl.concat(all_dfs)


df = pl.concat([
    # segm_df('elmcy_reevo'),
    # segm_df('meta_eoh'),
    segm_df('divrs_fs'),
])


In [7]:
# len(segm_ta._atoms) = 38058